# Classifying Penguins with Keras

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn import preprocessing
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, roc_curve

In [2]:
! pip install palmerpenguins
from palmerpenguins import load_penguins
penguins = load_penguins()
penguins = penguins.dropna()
penguins.head()


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,male,2007


In [3]:
#SHUFFLE THE DATA
penguins = penguins.sample(frac=1).reset_index(drop=True)
penguins.head()


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Biscoe,40.1,18.9,188.0,4300.0,male,2008
1,Chinstrap,Dream,54.2,20.8,201.0,4300.0,male,2008
2,Gentoo,Biscoe,42.7,13.7,208.0,3950.0,female,2008
3,Chinstrap,Dream,49.2,18.2,195.0,4400.0,male,2007
4,Adelie,Biscoe,41.4,18.6,191.0,3700.0,male,2008


In [4]:
penguins_x = pd.concat([penguins[['body_mass_g', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm']], pd.get_dummies(penguins['sex'])], axis = 1)
# penguins_x = penguins_x[['body_mass_g', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'female', 'male']]
penguins_x

,body_mass_g,bill_length_mm,bill_depth_mm,flipper_length_mm,female,male
0,4300.0,40.1,18.9,188.0,False,True
1,4300.0,54.2,20.8,201.0,False,True
2,3950.0,42.7,13.7,208.0,True,False
3,4400.0,49.2,18.2,195.0,False,True
4,3700.0,41.4,18.6,191.0,False,True
...,...,...,...,...,...,...
328,3875.0,41.4,18.5,202.0,False,True
329,4100.0,51.0,18.8,203.0,False,True
330,4650.0,45.5,13.7,214.0,True,False
331,3000.0,37.0,16.9,185.0,True,False


In [5]:
x = penguins_x.values
min_max_scaler = preprocessing.MinMaxScaler()
scaled_penguins_x = pd.DataFrame(min_max_scaler.fit_transform(x), columns=penguins_x.columns)
scaled_penguins_x

,body_mass_g,bill_length_mm,bill_depth_mm,flipper_length_mm,female,male
0,0.444444,0.290909,0.690476,0.271186,0.0,1.0
1,0.444444,0.803636,0.916667,0.491525,0.0,1.0
2,0.347222,0.385455,0.071429,0.610169,1.0,0.0
3,0.472222,0.621818,0.607143,0.389831,0.0,1.0
4,0.277778,0.338182,0.654762,0.322034,0.0,1.0
...,...,...,...,...,...,...
328,0.326389,0.338182,0.642857,0.508475,0.0,1.0
329,0.388889,0.687273,0.678571,0.525424,0.0,1.0
330,0.541667,0.487273,0.071429,0.711864,1.0,0.0
331,0.083333,0.178182,0.452381,0.220339,1.0,0.0


In [6]:
penguins_y = penguins['species']
print(penguins_y)
penguins_y = penguins_y.astype('category').cat.codes.to_numpy()
penguins_y

0         Adelie
1      Chinstrap
2         Gentoo
3      Chinstrap
4         Adelie
         ...    
328       Adelie
329    Chinstrap
330       Gentoo
331       Adelie
332       Adelie
Name: species, Length: 333, dtype: str


array([0, 1, 2, 1, 0, 2, 0, 0, 0, 1, 0, 1, 1, 2, 0, 0, 1, 1, 1, 2, 0, 0,
       2, 0, 0, 0, 0, 0, 0, 2, 2, 0, 1, 1, 0, 2, 2, 0, 1, 0, 1, 0, 2, 2,
       0, 1, 0, 0, 0, 2, 2, 1, 0, 2, 0, 2, 1, 2, 1, 2, 0, 1, 0, 2, 2, 2,
       2, 2, 2, 0, 0, 2, 1, 1, 2, 2, 1, 0, 0, 0, 2, 0, 1, 0, 2, 0, 2, 2,
       0, 0, 2, 0, 2, 0, 0, 1, 0, 2, 0, 2, 2, 2, 0, 0, 0, 0, 0, 2, 1, 2,
       1, 1, 2, 0, 2, 1, 0, 2, 0, 0, 1, 0, 0, 2, 2, 0, 2, 2, 0, 1, 2, 2,
       1, 2, 1, 0, 0, 0, 0, 0, 2, 0, 2, 0, 2, 1, 0, 2, 1, 0, 0, 1, 2, 0,
       0, 2, 2, 2, 2, 0, 0, 0, 0, 2, 2, 2, 1, 0, 0, 0, 0, 2, 2, 2, 0, 0,
       1, 0, 2, 0, 2, 2, 0, 0, 2, 0, 1, 2, 0, 1, 1, 2, 0, 0, 2, 1, 1, 0,
       0, 0, 0, 0, 2, 1, 1, 1, 0, 0, 0, 0, 2, 2, 2, 2, 2, 0, 0, 0, 1, 2,
       2, 1, 1, 1, 2, 0, 0, 0, 0, 0, 1, 2, 2, 2, 0, 0, 0, 2, 1, 0, 0, 0,
       1, 1, 0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 2, 1, 0, 1, 0, 2, 2,
       1, 2, 2, 2, 0, 1, 0, 1, 2, 2, 1, 1, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0,
       2, 2, 0, 0, 2, 0, 2, 2, 2, 1, 2, 1, 2, 2, 2,

In [7]:
#construct the model
inputs = keras.Input(shape=(6,))
x = layers.Dense(7, activation = 'relu')(inputs)
x = layers.Dense(5, activation = 'relu')(x)
x = layers.Dense(3, activation = 'relu')(x)
outputs = layers.Dense(3, activation='softmax')(x)
model = keras.Model(inputs=inputs, outputs=outputs, name="penguin_model")

In [8]:
model.summary()

Model: "penguin_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 6)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 7)              │            49 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │            40 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            18 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │            12 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 119 (476.00 B)

 Trainable params: 119 (476.00 B)

 Non-trainable params: 0 (0.00 B)

In [9]:
keras.utils.plot_model(model, show_shapes = True)

You must install pydot (`pip install pydot`) for `plot_model` to work.


In [10]:
model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.RMSprop(),
    metrics=["accuracy"],
)

history = model.fit(scaled_penguins_x, penguins_y, batch_size = 64, epochs=100, validation_split=0.1)

scores = model.evaluate(scaled_penguins_x, penguins_y, verbose=2)

Epoch 1/100


/Users/mkenne16/Documents/Advanced Machine Learning/week 7/.venv/lib/python3.11/site-packages/keras/src/backend/tensorflow/nn.py:1216: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step - accuracy: 0.4381 - loss: 1.2141 - val_accuracy: 0.4412 - val_loss: 1.2237
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5084 - loss: 1.1728 - val_accuracy: 0.3529 - val_loss: 1.1972
Epoch 3/100
1/5 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.4531 - loss: 1.1857

/Users/mkenne16/Documents/Advanced Machine Learning/week 7/.venv/lib/python3.11/site-packages/keras/src/backend/tensorflow/nn.py:1216: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.4515 - loss: 1.1511 - val_accuracy: 0.3529 - val_loss: 1.1777
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.4482 - loss: 1.1345 - val_accuracy: 0.3529 - val_loss: 1.1622
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.4482 - loss: 1.1207 - val_accuracy: 0.3529 - val_loss: 1.1482
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.4482 - loss: 1.1088 - val_accuracy: 0.3529 - val_loss: 1.1353
Epoch 7/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.4482 - loss: 1.0983 - val_accuracy: 0.3529 - val_loss: 1.1246
Epoch 8/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.4482 - loss: 1.0888 - val_accuracy: 0.3529 - val_loss: 1.1145
Epoch 9/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.4482 - loss: 1.0807 - val_accuracy: 0.3529 - val_loss: 1.1055
Epoch 10/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.4482 - loss: 1.0726 - val_accuracy: 0.3529 - val_loss: 1.0960
Epo

In [11]:
model_logit_true = keras.Model(inputs=inputs, outputs=outputs, name="penguin_model_scaled")

model_logit_true.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    optimizer=keras.optimizers.RMSprop(),
    metrics=["accuracy"],
)

history_logit_true = model_logit_true.fit(scaled_penguins_x, penguins_y, batch_size = 64, epochs = 100, validation_split = 0.1)

scores = model_logit_true.evaluate(scaled_penguins_x, penguins_y, verbose = 2)

Epoch 1/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.8997 - loss: 0.5641 - val_accuracy: 0.9412 - val_loss: 0.5584
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9164 - loss: 0.5535 - val_accuracy: 0.9412 - val_loss: 0.5502
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9264 - loss: 0.5456 - val_accuracy: 0.9412 - val_loss: 0.5433
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9298 - loss: 0.5394 - val_accuracy: 0.9412 - val_loss: 0.5354
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9298 - loss: 0.5331 - val_accuracy: 0.9412 - val_loss: 0.5292
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9298 - loss: 0.5273 - val_accuracy: 0.9412 - val_loss: 0.5233
Epoch 7/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9398 - loss: 0.5210 - val_accuracy: 0.9412 - val_loss: 0.5190
Epoch 8/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9398 - loss: 0.5157 - val_accuracy: 0.9412 - val_loss:

In [12]:
model_logit_true.predict(scaled_penguins_x)

11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 


array([[9.95033860e-01, 1.01200864e-03, 3.95405106e-03],
       [5.01229893e-03, 9.94987369e-01, 3.95297604e-07],
       [1.68231815e-01, 1.03896260e-01, 7.27871895e-01],
       [1.99769616e-01, 7.98085690e-01, 2.14463286e-03],
       [9.82876301e-01, 1.19698877e-02, 5.15391491e-03],
       [1.65044099e-01, 1.06453694e-01, 7.28502154e-01],
       [9.97649550e-01, 4.50266525e-04, 1.90011109e-03],
       [9.91859794e-01, 1.90873072e-03, 6.23157015e-03],
       [8.08156252e-01, 1.90789416e-01, 1.05439976e-03],
       [1.27293304e-01, 8.72645080e-01, 6.15698154e-05],
       [9.95612919e-01, 4.78455506e-04, 3.90868960e-03],
       [1.43781621e-02, 9.85620201e-01, 1.67481733e-06],
       [1.05582811e-02, 9.89437878e-01, 3.77241918e-06],
       [1.69009045e-01, 1.03284925e-01, 7.27706075e-01],
       [9.81998622e-01, 1.49540389e-02, 3.04741808e-03],
       [9.96967375e-01, 9.89786349e-04, 2.04291963e-03],
       [1.11651300e-02, 9.88832951e-01, 1.83644102e-06],
       [2.93145906e-02, 9.70629

In [13]:
penguins['species']

0         Adelie
1      Chinstrap
2         Gentoo
3      Chinstrap
4         Adelie
         ...    
328       Adelie
329    Chinstrap
330       Gentoo
331       Adelie
332       Adelie
Name: species, Length: 333, dtype: str

In [14]:
penguins_y

array([0, 1, 2, 1, 0, 2, 0, 0, 0, 1, 0, 1, 1, 2, 0, 0, 1, 1, 1, 2, 0, 0,
       2, 0, 0, 0, 0, 0, 0, 2, 2, 0, 1, 1, 0, 2, 2, 0, 1, 0, 1, 0, 2, 2,
       0, 1, 0, 0, 0, 2, 2, 1, 0, 2, 0, 2, 1, 2, 1, 2, 0, 1, 0, 2, 2, 2,
       2, 2, 2, 0, 0, 2, 1, 1, 2, 2, 1, 0, 0, 0, 2, 0, 1, 0, 2, 0, 2, 2,
       0, 0, 2, 0, 2, 0, 0, 1, 0, 2, 0, 2, 2, 2, 0, 0, 0, 0, 0, 2, 1, 2,
       1, 1, 2, 0, 2, 1, 0, 2, 0, 0, 1, 0, 0, 2, 2, 0, 2, 2, 0, 1, 2, 2,
       1, 2, 1, 0, 0, 0, 0, 0, 2, 0, 2, 0, 2, 1, 0, 2, 1, 0, 0, 1, 2, 0,
       0, 2, 2, 2, 2, 0, 0, 0, 0, 2, 2, 2, 1, 0, 0, 0, 0, 2, 2, 2, 0, 0,
       1, 0, 2, 0, 2, 2, 0, 0, 2, 0, 1, 2, 0, 1, 1, 2, 0, 0, 2, 1, 1, 0,
       0, 0, 0, 0, 2, 1, 1, 1, 0, 0, 0, 0, 2, 2, 2, 2, 2, 0, 0, 0, 1, 2,
       2, 1, 1, 1, 2, 0, 0, 0, 0, 0, 1, 2, 2, 2, 0, 0, 0, 2, 1, 0, 0, 0,
       1, 1, 0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 2, 1, 0, 1, 0, 2, 2,
       1, 2, 2, 2, 0, 1, 0, 1, 2, 2, 1, 1, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0,
       2, 2, 0, 0, 2, 0, 2, 2, 2, 1, 2, 1, 2, 2, 2,